In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load main dataset
app_train = pd.read_csv('../data/application_train.csv')

print(f"Shape: {app_train.shape}")
print(f"\nTarget distribution:")
print(app_train['TARGET'].value_counts())
print(f"\nDefault rate: {app_train['TARGET'].mean():.2%}")

Shape: (307511, 122)

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default rate: 8.07%


In [2]:
# Quick overview of the dataset
print("=== Basic Info ===")
print(f"Total applications: {len(app_train):,}")
print(f"Total features: {app_train.shape[1]}")
print(f"\n=== Missing Values (Top 20) ===")
missing = app_train.isnull().sum()
missing_pct = (missing / len(app_train) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0].head(20))

print(f"\n=== Data Types ===")
print(app_train.dtypes.value_counts())

=== Basic Info ===
Total applications: 307,511
Total features: 122

=== Missing Values (Top 20) ===
                          Missing Count  Missing %
COMMONAREA_MEDI                  214865      69.87
COMMONAREA_AVG                   214865      69.87
COMMONAREA_MODE                  214865      69.87
NONLIVINGAPARTMENTS_MODE         213514      69.43
NONLIVINGAPARTMENTS_AVG          213514      69.43
NONLIVINGAPARTMENTS_MEDI         213514      69.43
FONDKAPREMONT_MODE               210295      68.39
LIVINGAPARTMENTS_MODE            210199      68.35
LIVINGAPARTMENTS_AVG             210199      68.35
LIVINGAPARTMENTS_MEDI            210199      68.35
FLOORSMIN_AVG                    208642      67.85
FLOORSMIN_MODE                   208642      67.85
FLOORSMIN_MEDI                   208642      67.85
YEARS_BUILD_MEDI                 204488      66.50
YEARS_BUILD_MODE                 204488      66.50
YEARS_BUILD_AVG                  204488      66.50
OWN_CAR_AGE                      

## Notebook 1 — Data Loading

When I first opened this dataset I was genuinely surprised by the scale of it.
307,511 real loan applications from Home Credit, a fintech company operating
across Southeast Asia and Eastern Europe. Each application has 122 features
covering everything from income and employment to property ownership and
external credit scores.

The first thing I checked was the default rate — only 8.07% of applicants
actually defaulted on their loans. This is called class imbalance, and it's
one of the most important challenges in credit risk modeling. A naive model
that simply predicts "no default" for everyone would be 92% accurate but
completely useless for a lender. Handling this imbalance properly is what
separates a real credit risk model from a toy example.

## Missing Values Analysis

The missing value analysis revealed something interesting — a large group of
property-related features like COMMONAREA, NONLIVINGAPARTMENTS, and FLOORSMIN
are missing for nearly 70% of applicants. This makes sense when you think about
it: many applicants simply don't own property, so these fields were never filled.

The dataset has 65 float columns, 41 integer columns, and 16 categorical
columns. The mix of data types tells me this will need careful preprocessing
before any model can use it — encoding categoricals, imputing missing values,
and deciding which columns to drop entirely.## Missing Values Analysis

The missing value analysis revealed something interesting — a large group of
property-related features like COMMONAREA, NONLIVINGAPARTMENTS, and FLOORSMIN
are missing for nearly 70% of applicants. This makes sense when you think about
it: many applicants simply don't own property, so these fields were never filled.

The dataset has 65 float columns, 41 integer columns, and 16 categorical
columns. The mix of data types tells me this will need careful preprocessing
before any model can use it — encoding categoricals, imputing missing values,
and deciding which columns to drop entirely.

In [3]:
# Look at key features for credit risk
key_features = [
    'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'AMT_GOODS_PRICE', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

print("=== Key Features Overview ===")
print(app_train[key_features].describe().round(2))
print(f"\n=== Categorical Features ===")
cat_features = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 
                'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']
for col in cat_features:
    print(f"\n{col}:")
    print(app_train[col].value_counts())

=== Key Features Overview ===
          TARGET  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE  \
count  307511.00      3.075110e+05   307511.00    307499.00        307233.00   
mean        0.08      1.687979e+05   599026.00     27108.57        538396.21   
std         0.27      2.371231e+05   402490.78     14493.74        369446.46   
min         0.00      2.565000e+04    45000.00      1615.50         40500.00   
25%         0.00      1.125000e+05   270000.00     16524.00        238500.00   
50%         0.00      1.471500e+05   513531.00     24903.00        450000.00   
75%         0.00      2.025000e+05   808650.00     34596.00        679500.00   
max         1.00      1.170000e+08  4050000.00    258025.50       4050000.00   

       CNT_CHILDREN  DAYS_BIRTH  DAYS_EMPLOYED  DAYS_REGISTRATION  \
count     307511.00   307511.00      307511.00          307511.00   
mean           0.42   -16037.00       63815.05           -4986.12   
std            0.72     4363.99      14127

## Key Features Overview

A few things stood out immediately when I looked at the key features.

The average loan amount is around 599,000 while the average income is only
168,000 — meaning most people are borrowing roughly 3.5 times their annual
income. That's a significant debt burden and likely one of the strongest
predictors of default risk.

Female applicants outnumber male applicants almost 2 to 1 (202k vs 105k),
which is an interesting demographic pattern worth exploring in the EDA.
The majority of applicants are working class with secondary education,
which reflects the target market of a consumer lending fintech.

The DAYS_BIRTH and DAYS_EMPLOYED columns are stored as negative numbers
counting backwards from the application date. I'll need to convert these
to actual age and employment duration in the cleaning notebook.

The EXT_SOURCE columns (1, 2, 3) appear to be external credit scores
normalized between 0 and 1. Based on domain knowledge these are typically
among the strongest predictors of credit default and I expect them to
dominate the feature importance rankings in the final model.

In [ ]:
# Save key observations
print("=== Summary for Notebook 1 ===")
print(f"Dataset: {app_train.shape[0]:,} loan applications")
print(f"Features: {app_train.shape[1]} columns")
print(f"Default rate: {app_train['TARGET'].mean():.2%}")
print(f"Gender split: {app_train['CODE_GENDER'].value_counts().to_dict()}")
print(f"Avg income: {app_train['AMT_INCOME_TOTAL'].mean():,.0f}")
print(f"Avg loan amount: {app_train['AMT_CREDIT'].mean():,.0f}")
print(f"Columns with >50% missing: {(app_train.isnull().mean() > 0.5).sum()}")

# Save to data folder
app_train.to_csv('../data/app_train_raw.csv', index=False)
print("\nRaw data saved!")

=== Summary for Notebook 1 ===
Dataset: 307,511 loan applications
Features: 122 columns
Default rate: 8.07%
Gender split: {'F': 202448, 'M': 105059, 'XNA': 4}
Avg income: 168,798
Avg loan amount: 599,026
Columns with >50% missing: 41


## Summary

This initial exploration gave me a clear picture of what I'm working with.
The dataset is large enough to build a reliable model, the default rate
creates a meaningful class imbalance challenge, and there are some very
promising features like the external credit scores that I expect will
drive most of the model's predictive power.

The next step is cleaning — dropping high-missing columns, encoding
categoricals, engineering new features, and handling the class imbalance
using SMOTE oversampling before modeling.